In [0]:
# ===========================================
# Notebook Name:
# 01_explore_meta
#
# Purpose:
# Explore tournament and deck data
# stored in Silver tables.
#
# Input:
# pokemon.silver.tournament_results
# pokemon.silver.decks
# pokemon.silver.deck_cards
#
# Output:
# Exploratory tables and charts
# ===========================================

In [0]:
from pyspark.sql import functions as F

TOURNAMENT_RESULTS_TABLE = (
    "pokemon.silver.tournament_results"
)

DECKS_TABLE = "pokemon.silver.decks"

DECK_CARDS_TABLE = (
    "pokemon.silver.deck_cards"
)

tournament_results_df = spark.table(
    TOURNAMENT_RESULTS_TABLE
)

decks_df = spark.table(DECKS_TABLE)

deck_cards_df = spark.table(
    DECK_CARDS_TABLE
)

In [0]:
print(
    "Tournament results:",
    tournament_results_df.count(),
)

print(
    "Decks:",
    decks_df.count(),
)

print(
    "Deck card rows:",
    deck_cards_df.count(),
)

In [0]:
display(
    decks_df
    .groupBy("is_valid")
    .agg(
        F.count("*").alias("deck_count")
    )
)

In [0]:
display(
    deck_cards_df
    .groupBy("deck_code")
    .agg(
        F.sum("quantity").alias(
            "total_cards"
        )
    )
    .orderBy("deck_code")
)

In [0]:
ranked_decks_df = (
    tournament_results_df
    .join(
        decks_df,
        on="deck_code",
        how="left",
    )
    .select(
        "tournament_id",
        "rank",
        "player_name",
        "deck_code",
        "total_cards",
        "unique_card_names",
        "card_type_rows",
        "is_valid",
    )
)

display(
    ranked_decks_df.orderBy(
        "tournament_id",
        "rank",
    )
)

In [0]:
total_deck_count = decks_df.select(
    "deck_code"
).distinct().count()

print(
    "Total deck count:",
    total_deck_count,
)

In [0]:
card_usage_df = (
    deck_cards_df
    .groupBy(
        "card_name",
        "category",
    )
    .agg(
        F.countDistinct(
            "deck_code"
        ).alias("decks_using_card"),
        F.sum("quantity").alias(
            "total_copies"
        ),
        F.avg("quantity").alias(
            "avg_copies_when_used"
        ),
    )
    .withColumn(
        "inclusion_rate",
        F.col("decks_using_card")
        / F.lit(total_deck_count),
    )
    .orderBy(
        F.col("decks_using_card").desc(),
        F.col("total_copies").desc(),
    )
)

display(card_usage_df)

In [0]:
display(
    card_usage_df.select(
        "card_name",
        "category",
        "decks_using_card",
        F.round(
            F.col("inclusion_rate") * 100,
            1,
        ).alias("inclusion_rate_pct"),
        "total_copies",
        F.round(
            "avg_copies_when_used",
            2,
        ).alias("avg_copies_when_used"),
    )
    .limit(20)
)

In [0]:
category_summary_df = (
    deck_cards_df
    .groupBy("category")
    .agg(
        F.sum("quantity").alias(
            "total_card_copies"
        ),
        F.countDistinct(
            "deck_code"
        ).alias("deck_count"),
        F.countDistinct(
            "card_name"
        ).alias("unique_card_names"),
    )
    .withColumn(
        "avg_cards_per_deck",
        F.col("total_card_copies")
        / F.lit(total_deck_count),
    )
    .orderBy(
        F.col("total_card_copies").desc()
    )
)

display(category_summary_df)

In [0]:
deck_category_df = (
    deck_cards_df
    .groupBy(
        "deck_code",
        "category",
    )
    .agg(
        F.sum("quantity").alias(
            "category_cards"
        )
    )
)

display(
    deck_category_df.orderBy(
        "deck_code",
        "category",
    )
)

In [0]:
deck_category_pivot_df = (
    deck_category_df
    .groupBy("deck_code")
    .pivot("category")
    .sum("category_cards")
    .fillna(0)
)

display(deck_category_pivot_df)

In [0]:
ranked_category_df = (
    tournament_results_df
    .select(
        "deck_code",
        "rank",
        "player_name",
    )
    .join(
        deck_category_pivot_df,
        on="deck_code",
        how="left",
    )
    .orderBy("rank")
)

display(ranked_category_df)

In [0]:
TOP_N = 4

top_deck_codes_df = (
    tournament_results_df
    .filter(F.col("rank") <= TOP_N)
    .select("deck_code")
    .distinct()
)

top_deck_count = top_deck_codes_df.count()

print(
    "Top deck count:",
    top_deck_count,
)

In [0]:
top_card_usage_df = (
    deck_cards_df
    .join(
        top_deck_codes_df,
        on="deck_code",
        how="inner",
    )
    .groupBy(
        "card_name",
        "category",
    )
    .agg(
        F.countDistinct(
            "deck_code"
        ).alias("top_decks_using_card"),
        F.sum("quantity").alias(
            "top_total_copies"
        ),
    )
    .withColumn(
        "top_inclusion_rate",
        F.col("top_decks_using_card")
        / F.lit(top_deck_count),
    )
    .orderBy(
        F.col("top_decks_using_card").desc(),
        F.col("top_total_copies").desc(),
    )
)

display(top_card_usage_df)

In [0]:
usage_comparison_df = (
    card_usage_df.alias("all")
    .join(
        top_card_usage_df.alias("top"),
        on=[
            "card_name",
            "category",
        ],
        how="left",
    )
    .select(
        "card_name",
        "category",
        F.round(
            F.col("all.inclusion_rate") * 100,
            1,
        ).alias("all_inclusion_pct"),
        F.round(
            F.coalesce(
                F.col(
                    "top.top_inclusion_rate"
                ),
                F.lit(0),
            ) * 100,
            1,
        ).alias("top4_inclusion_pct"),
    )
    .withColumn(
        "top4_lift_pct_point",
        F.col("top4_inclusion_pct")
        - F.col("all_inclusion_pct"),
    )
    .orderBy(
        F.col(
            "top4_lift_pct_point"
        ).desc()
    )
)

display(usage_comparison_df)

In [0]:
deck_card_presence_df = (
    deck_cards_df
    .select(
        "deck_code",
        "card_name",
    )
    .dropDuplicates()
)

In [0]:
deck_pair_similarity_df = (
    deck_card_presence_df.alias("left")
    .join(
        deck_card_presence_df.alias("right"),
        on=(
            F.col("left.card_name")
            == F.col("right.card_name")
        ),
        how="inner",
    )
    .filter(
        F.col("left.deck_code")
        < F.col("right.deck_code")
    )
    .groupBy(
        F.col("left.deck_code").alias(
            "deck_code_a"
        ),
        F.col("right.deck_code").alias(
            "deck_code_b"
        ),
    )
    .agg(
        F.countDistinct(
            "left.card_name"
        ).alias("shared_card_names")
    )
    .orderBy(
        F.col("shared_card_names").desc()
    )
)

display(deck_pair_similarity_df)

In [0]:
deck_rank_df = (
    tournament_results_df
    .select(
        "deck_code",
        "rank",
    )
)

deck_pair_ranked_df = (
    deck_pair_similarity_df
    .join(
        deck_rank_df.alias("rank_a"),
        F.col("deck_code_a")
        == F.col(
            "rank_a.deck_code"
        ),
        "left",
    )
    .join(
        deck_rank_df.alias("rank_b"),
        F.col("deck_code_b")
        == F.col(
            "rank_b.deck_code"
        ),
        "left",
    )
    .select(
        "deck_code_a",
        F.col("rank_a.rank").alias(
            "rank_a"
        ),
        "deck_code_b",
        F.col("rank_b.rank").alias(
            "rank_b"
        ),
        "shared_card_names",
    )
    .orderBy(
        F.col("shared_card_names").desc()
    )
)

display(deck_pair_ranked_df)

In [0]:
%sql

SELECT
    card_name,
    category,
    COUNT(DISTINCT deck_code) AS decks_using_card,
    SUM(quantity) AS total_copies,
    ROUND(AVG(quantity), 2) AS avg_copies_when_used
FROM pokemon.silver.deck_cards
GROUP BY
    card_name,
    category
ORDER BY
    decks_using_card DESC,
    total_copies DESC
LIMIT 20;

In [0]:
%sql

SELECT
    tr.rank,
    tr.player_name,
    dc.card_name,
    dc.quantity,
    dc.category
FROM pokemon.silver.tournament_results AS tr
INNER JOIN pokemon.silver.deck_cards AS dc
    ON tr.deck_code = dc.deck_code
WHERE tr.rank <= 4
ORDER BY
    tr.rank,
    dc.category,
    dc.card_name;

In [0]:
analysis_deck_cards_df = (
    deck_cards_df
    .filter(
        ~(
            (F.col("category") == "energy")
            & F.col("card_name").startswith("基本")
        )
    )
)

In [0]:
card_usage_df = (
    analysis_deck_cards_df
    .groupBy(
        "card_name",
        "category",
    )
    .agg(
        F.countDistinct("deck_code").alias(
            "decks_using_card"
        ),
        F.sum("quantity").alias(
            "total_copies"
        ),
        F.avg("quantity").alias(
            "avg_copies_when_used"
        ),
    )
    .withColumn(
        "inclusion_rate",
        F.col("decks_using_card")
        / F.lit(total_deck_count),
    )
    .orderBy(
        F.col("decks_using_card").desc(),
        F.col("total_copies").desc(),
    )
)

In [0]:
pokemon_presence_df = (
    deck_cards_df
    .filter(F.col("category") == "pokemon")
    .select(
        "deck_code",
        "card_name",
    )
    .dropDuplicates()
)

In [0]:
pokemon_similarity_df = (
    pokemon_presence_df.alias("left")
    .join(
        pokemon_presence_df.alias("right"),
        F.col("left.card_name")
        == F.col("right.card_name"),
        "inner",
    )
    .filter(
        F.col("left.deck_code")
        < F.col("right.deck_code")
    )
    .groupBy(
        F.col("left.deck_code").alias(
            "deck_code_a"
        ),
        F.col("right.deck_code").alias(
            "deck_code_b"
        ),
    )
    .agg(
        F.countDistinct(
            "left.card_name"
        ).alias("shared_pokemon_names")
    )
    .orderBy(
        F.col("shared_pokemon_names").desc()
    )
)

display(pokemon_similarity_df)